In [0]:
# Step 1: Import Required Libraries
import pyspark
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# Step 2: Load Dataset
df = spark.table("workspace.default.dataset")

In [0]:
# Step 3: Explore the Dataset

df.printSchema()
print("Total number of Records:", df.count())
df.describe().show()
df.show(5)

root
 |-- user_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- store_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- age: long (nullable = true)
 |-- subscription: string (nullable = true)
 |-- status: string (nullable = true)

Total number of Records: 10200
+-------+-------+--------+-------+------+----------------+------------------+------------------+------------+---------+
|summary|user_id|store_id|   city|region|product_category|             price|               age|subscription|   status|
+-------+-------+--------+-------+------+----------------+------------------+------------------+------------+---------+
|  count|  10200|   10200|  10200| 10200|           10200|              3456|             10200|       10200|     6879|
|   mean|   NULL|    NULL|   NULL|  NULL|            NULL| 504.6765740740734|30.5503

In [0]:
# Step 4: Data Cleaning

# Remove Duplicate Records
df = df.dropDuplicates(["user_id","transaction_date"])
print("Records after removing duplicates:", df.count())

# Handle Missing Values
df = df.na.fill({"price":0, "status":"Unknown"})

Records after removing duplicates: 9917


In [0]:
# Step 5: Filter the Dataset

# Users from West Region
west_df = df.filter(col("region")=="West")
display(west_df)

# Premium Users between 18 and 30 Years
premium_df = df.filter((col("age").between(18,30)) & (col("subscription")=="Premium"))
display(premium_df)

user_id,transaction_date,store_id,city,region,product_category,price,age,subscription,status
U2620,2024-01-29,S008,Phoenix,West,Clothing,238.75,19,Premium,Unknown
U1729,2024-01-09,S008,Los Angeles,West,Electronics,0.0,33,Premium,Unknown
U2800,2024-06-14,S006,Phoenix,West,Furniture,0.0,21,Basic,Pending
U1644,2024-03-09,S023,Los Angeles,West,Furniture,328.38,31,Basic,Unknown
U0048,2024-06-23,S025,Phoenix,West,Furniture,281.49,26,Premium,Pending
U2324,2024-01-21,S003,Phoenix,West,Clothing,978.42,40,Premium,Completed
U0129,2024-03-25,S009,Phoenix,West,Electronics,0.0,33,Premium,Unknown
U2983,2024-03-27,S007,Los Angeles,West,Electronics,0.0,29,Premium,Pending
U1895,2024-03-04,S004,San Diego,West,Furniture,69.58,42,Premium,Completed
U0964,2024-04-13,S019,Phoenix,West,Electronics,602.63,18,Basic,Unknown


user_id,transaction_date,store_id,city,region,product_category,price,age,subscription,status
U2620,2024-01-29,S008,Phoenix,West,Clothing,238.75,19,Premium,Unknown
U1732,2024-03-28,S025,Seattle,North,Electronics,0.0,28,Premium,Pending
U2534,2024-04-02,S002,Chicago,East,Electronics,0.0,18,Premium,Completed
U0048,2024-06-23,S025,Phoenix,West,Furniture,281.49,26,Premium,Pending
U2173,2024-01-01,S004,Denver,North,Electronics,0.0,23,Premium,Completed
U1735,2024-04-15,S022,Miami,South,Electronics,660.39,19,Premium,Pending
U2983,2024-03-27,S007,Los Angeles,West,Electronics,0.0,29,Premium,Pending
U0778,2024-03-16,S024,New York,East,Furniture,551.35,26,Premium,Completed
U2752,2024-06-14,S025,Denver,North,Clothing,0.0,30,Premium,Completed
U2203,2024-02-24,S003,Seattle,North,Clothing,0.0,25,Premium,Pending


In [0]:
# Step 6: Transform the Dataset

# Rename Column
df = df.withColumnRenamed("price","product_price")

# Change Data Type
df = df.withColumn("product_price", col("product_price").cast(DoubleType()))

df.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- store_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_price: double (nullable = false)
 |-- age: long (nullable = true)
 |-- subscription: string (nullable = true)
 |-- status: string (nullable = false)



In [0]:
# Step 7: Aggregate the Dataset

df.select(
    count("*").alias("Total Records"),
    min("product_price").alias("Minimum Price"),
    max("product_price").alias("Maximum Price"),
    avg("product_price").alias("Average Price"),
    sum("product_price").alias("Total Price")
).show()

+-------------+-------------+-------------+------------------+------------------+
|Total Records|Minimum Price|Maximum Price|     Average Price|       Total Price|
+-------------+-------------+-------------+------------------+------------------+
|         9917|          0.0|       999.66|170.91038015528872|1694918.2399999981|
+-------------+-------------+-------------+------------------+------------------+



In [0]:
# Step 8: Grouping the dataset using aggregate functions

#Sales summary of each store
stores = df.groupBy("store_id").agg( count("*").alias("Transactions"), sum("product_price").alias("Revenue"), avg("product_price").alias("Average Price") ) 
display(stores)

#Avg product price by product category
categories = df.groupBy("product_category").agg(count("*").alias("Products"), avg("product_price").alias("Average Price"))
display(categories)

store_id,Transactions,Revenue,Average Price
S008,383,71642.93999999997,187.0572845953002
S019,380,62214.53000000003,163.72244736842111
S025,408,74073.71999999999,181.55323529411763
S015,400,57191.120000000024,142.97780000000006
S002,384,66791.19999999998,173.9354166666666
S006,376,58763.630000000005,156.28625000000002
S023,379,70032.65999999999,184.78274406332451
S024,423,71496.18000000001,169.02170212765958
S016,409,64773.590000000026,158.37063569682158
S013,405,69546.31,171.71928395061727


product_category,Products,Average Price
Clothing,3325,170.47329924812027
Electronics,3201,174.94243986254295
Furniture,3391,167.5328133294011


In [0]:


# Step 9: Complete Data Processing Pipeline with all the above steps

final_df = (

    spark.table("workspace.default.dataset")
    .dropDuplicates(["user_id","transaction_date"])

    .na.fill({
        "price":0,
        "status":"Unknown"
    })

    .filter(col("region")=="West")

    .withColumn(
        "price",
        col("price").cast(DoubleType())
    )

    .groupBy("store_id")

    .agg(
        count("*").alias("Transactions"),
        sum("price").alias("Revenue"),
        avg("price").alias("Average Price"),
        min("price").alias("Minimum Price"),
        max("price").alias("Maximum Price")
    )
)

In [0]:
# Step 10: Displaying final output of the pipeline (clean dataset)
display(final_df)

store_id,Transactions,Revenue,Average Price,Minimum Price,Maximum Price
S008,91,14449.089999999997,158.78120879120874,0.0,998.18
S006,102,17660.73,173.14441176470586,0.0,980.81
S023,82,14395.02,175.54902439024391,0.0,911.22
S025,124,21474.879999999997,173.18451612903223,0.0,911.14
S003,91,16586.46,182.26879120879119,0.0,996.09
S009,80,12531.99,156.649875,0.0,967.73
S007,91,14284.830000000005,156.9761538461539,0.0,952.63
S004,102,16425.43,161.0336274509804,0.0,962.81
S019,93,9984.789999999999,107.36333333333333,0.0,969.98
S005,101,14251.629999999997,141.10524752475246,0.0,969.02
